In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist
from IPython.display import display, HTML

In [ ]:
# Define the Raw Data URLs
batter_url = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSSituationalSplits-Batters-FullSeason.csv"
catcher_url = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSSituationalSplits-Catchers-FullSeason.csv"

In [ ]:
# Create function to execute clustering pipeline
def run_clustering_pipeline(url_path, position_label, min_opportunities=15):
    print(f"\n==================================================================")
    print(f"🚀 RUNNING COMPREHENSIVE CLUSTERING PIPELINE FOR: {position_label.upper()}")
    print(f"==================================================================")

    # Load Dataset
    try:
        df = pd.read_csv(url_path, encoding="latin1")
        print(f"Successfully connected and loaded {len(df)} rows from GitHub.")
    except Exception as e:
        print(f"Error fetching data: {e}")
        return

    # Filter out players with tiny sample sizes to remove statistical noise
    df_filtered = df[df["Total_Opp_Ovr"] >= min_opportunities].copy().reset_index(drop=True)
    print(f"Rows after filtering (Total_Opp_Ovr >= {min_opportunities}): {len(df_filtered)}")

    # ISOLATION RULE: Keep columns ending in _Rate, but strictly block anything containing 'Ovr' or 'Total'
    rate_features = [col for col in df_filtered.columns if col.endswith("_Rate")]
    rate_features = [col for col in rate_features if "Ovr" not in col and "Total" not in col]

    print(f"Isolating all {len(rate_features)} aggression features for clustering calculation:")

    # Replace NAs with 0.0
    X = df_filtered[rate_features].fillna(0.0)

    # Standardize Features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Get screenshot of final data used for model
    scaled_df = pd.DataFrame(data=X_scaled, columns=rate_features)
    print(scaled_df.head())

    # ---- K-MEANS WITH MULTIPLE K-VALUES (k=2-7) ----
    k_values = [2, 3, 4, 5, 6, 7]
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes_flat = axes.flatten()
    kmeans_results = {}

    for i, k in enumerate(k_values):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(X_scaled)
        kmeans_results[k] = cluster_labels

        # Calculate silhouette diagnostics
        avg_silhouette = silhouette_score(X_scaled, cluster_labels)
        sample_silhouette_values = silhouette_samples(X_scaled, cluster_labels)

        ax = axes_flat[i]
        y_lower = 10
        for j in range(k):
            ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == j]
            ith_cluster_silhouette_values.sort()

            size_cluster_i = ith_cluster_silhouette_values.shape[0]
            y_upper = y_lower + size_cluster_i

            color = plt.cm.nipy_spectral(float(j) / k)
            ax.fill_betweenx(np.arange(y_lower, y_upper), 0, ith_cluster_silhouette_values,
                             facecolor=color, edgecolor=color, alpha=0.7)
            ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(j))
            y_lower = y_upper + 10

        ax.set_title(f"k = {k} | Avg Score: {avg_silhouette:.3f}", fontsize=11, fontweight="bold")
        ax.set_xlabel("Silhouette Coefficients")
        ax.set_ylabel("Cluster Profiles")
        ax.axvline(x=avg_silhouette, color="red", linestyle="--")
        ax.set_xlim([-0.1, 1.0])

    plt.suptitle(f"{position_label} Pure Aggression Splits: Silhouette Analysis Profiles", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

    # Identify the highest scoring k to map out a representative profile
    best_k = max(k_values, key=lambda k: silhouette_score(X_scaled, kmeans_results[k]))
    df_filtered["KMeans_Cluster"] = kmeans_results[best_k]
    print(f"ℹ️ Silhouette Method flags k={best_k} as the highest average score mapping.")

    # ---- K-MEANS SCATTER REPRESENTATION ----
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df_filtered, x="Total_ChallengeRate", y="Total_OverturnedRate", hue="KMeans_Cluster", palette="Set1", s=80, alpha=0.8)
    plt.title(f"{position_label} K-Means Cluster Profiling (Optimal k={best_k})", fontsize=12, fontweight="bold")
    plt.xlabel("Overall Challenge Rate")
    plt.ylabel("Overturn Rate")
    plt.legend(title="Cluster")
    plt.tight_layout()
    plt.show()

    # ---- HIERARCHICAL CLUSTERING WITH COSINE DISTANCE ----
    distance_cosine = pdist(X_scaled, metric="cosine")
    linked_tree = linkage(distance_cosine, method="average")

    plt.figure(figsize=(12, 6))
    dendrogram(linked_tree,
               labels=df_filtered["lastName"].values,
               truncate_mode="lastp",
               p=25,
               leaf_rotation=90,
               leaf_font_size=9,
               show_contracted=True)

    plt.title(f"Hierarchical Tree for {position_label} (Cosine Similarity Linkage)", fontsize=12, fontweight="bold")
    plt.xlabel("Aggregated Player Cluster Nodes")
    plt.ylabel("Cosine Distance Matrix Cutoff")
    plt.tight_layout()
    plt.show()

    # ---- EVALUATING DISCOVERED CLUSTERS ----
    print(f"\n📊 GENRE-WIDE ARCHETYPE PROFILES (Complete Aggression Splits vs. Target Overturn Success):")

    # Dynamically build the summary feature list:
    # Starts with player metadata, appends the success rates, and then unpacks ALL 19 clustering rate splits
    comprehensive_features = ["Total_ChallengeRate", "Total_OverturnedRate"] + rate_features

    # Protect against any sudden column schema naming changes in the source branch files
    comprehensive_features = [f for f in comprehensive_features if f in df_filtered.columns]

    # Aggregate data rows by cluster group
    cluster_profile_table = df_filtered.groupby("KMeans_Cluster")[comprehensive_features].mean()

    # Prepend the total group size volume indicator
    cluster_profile_table.insert(0, "Player_Count", df_filtered.groupby("KMeans_Cluster").size())

    # Transpose the matrix so that clusters run as columns and ALL rates display as row definitions
    transposed_table = cluster_profile_table.T.round(3)

    # Format the header labels cleanly for web deployment
    transposed_table.columns = [f"Cluster {c}" for c in transposed_table.columns]

    # Convert whole compiled dataframe block directly into styled HTML code elements
    html_markup = transposed_table.to_html(classes="table table-striped table-hover", border=1)

    print("\n--- COPY AND PASTE THIS ENTIRE TRANSPOSED HTML CODE INTO YOUR CLUSTERING.HTML WEB TAB ---")
    print(html_markup)
    print("-----------------------------------------------------------------\n")

    display(HTML(f"<h3>Comprehensive Transposed Aggression Matrix: {position_label}</h3>"))
    display(HTML(html_markup))

In [ ]:
# Process batters dataset through the clustering pipeline
run_clustering_pipeline(url_path=batter_url, position_label="Batters", min_opportunities=100)

In [ ]:
# Process catchers dataset through the clustering pipeline
run_clustering_pipeline(url_path=catcher_url, position_label="Catchers", min_opportunities=100)

# Create diagrams for website

In [ ]:
# Establish structural figure configuration style
#plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available() else "default")
#plt.rcParams["font.sans-serif"] = "Liberation Sans"
#plt.rcParams["axes.edgecolor"] = "#ccc"

# ==============================================================================
# DIAGRAM 1: PARTITIONAL (K-MEANS) VS HIERARCHICAL THEORY
# ==============================================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Generate mock data blobs for visualization mapping
np.random.seed(42)
b1 = np.random.normal(loc=[2, 3], scale=0.4, size=(15, 2))
b2 = np.random.normal(loc=[5, 5], scale=0.4, size=(15, 2))
b3 = np.random.normal(loc=[4, 1], scale=0.4, size=(15, 2))
data_points = np.vstack([b1, b2, b3])

# Left Chart: Partitional K-Means Spatial Partition Boundaries
ax1.scatter(b1[:, 0], b1[:, 1], color="crimson", edgecolors="black", s=60, label="Cluster A", alpha=0.85)
ax1.scatter(b2[:, 0], b2[:, 1], color="royalblue", edgecolors="black", s=60, label="Cluster B", alpha=0.85)
ax1.scatter(b3[:, 0], b3[:, 1], color="forestgreen", edgecolors="black", s=60, label="Cluster C", alpha=0.85)

# Plot explicit partitional centroids
ax1.scatter([2.0, 4.9, 4.0], [2.9, 4.9, 1.1], color="yellow", edgecolors="black", s=180, marker="*", label="Centroids", zorder=10)
ax1.set_title("Partitional Architecture (K-Means Spaces)", fontsize=12, fontweight="bold", pad=10)
ax1.set_xlabel("Feature Axis X")
ax1.set_ylabel("Feature Axis Y")
ax1.legend(loc="upper left", frameon=True)
ax1.grid(True, linestyle="--", alpha=0.5)

# Right Chart: Hierarchical Tree Architecture (Dendrogram)
# Samples a tiny fraction of data lines to produce a highly readable conceptual tree
linked = linkage(data_points[:12], method="ward")
dendrogram(linked, ax=ax2, orientation="top", color_threshold=3.5,
           above_threshold_color="#444444")
ax2.set_title("Hierarchical Tree Architecture (Dendrogram)", fontsize=12, fontweight="bold", pad=10)
ax2.set_xlabel("Individual Data Point Indices")
ax2.set_ylabel("Linkage Proximity / Node Dissimilarity")
ax2.grid(axis="y", linestyle="--", alpha=0.5)

plt.suptitle("Conceptual Framework: Partitional vs. Hierarchical Space Construction", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("placeholder_cluster_theory1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ==============================================================================
# DIAGRAM 2: EUCLIDEAN DISTANCE VS COSINE SIMILARITY
# ==============================================================================
plt.figure(figsize=(8, 7))
ax3 = plt.subplot(111)

# Define two vector configurations with similar directions but wildly different magnitudes
vA = np.array([2, 5])
vB = np.array([5, 12.5])  # Completely collinear directions (Angle = 0)
vC = np.array([6, 2])     # Divergent direction

# Draw vector paths from geometric origin point (0,0)
ax3.quiver(0, 0, vA[0], vA[1], angles="xy", scale_units="xy", scale=1, color="crimson", width=0.007, label="Vector A (Player 1 Profile)")
ax3.quiver(0, 0, vB[0], vB[1], angles="xy", scale_units="xy", scale=1, color="royalblue", width=0.007, label="Vector B (Player 2 Profile - High Volume)")
ax3.quiver(0, 0, vC[0], vC[1], angles="xy", scale_units="xy", scale=1, color="forestgreen", width=0.007, label="Vector C (Player 3 Profile - Alternate Strategy)")

# Draw a physical straight line representing Euclidean Distance between A and B
ax3.plot([vA[0], vB[0]], [vA[1], vB[1]], color="dimgray", linestyle=":", linewidth=2, marker="o")
ax3.text((vA[0]+vB[0])/2 + 0.2, (vA[1]+vB[1])/2, "Euclidean Distance\n(Measures Magnitude Delta)", color="dimgray", fontsize=9, fontweight="semibold", bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7))

# Draw arc marker to show Cosine Angular Divergence between Vector B and Vector C
theta = np.linspace(np.arctan2(vC[1], vC[0]), np.arctan2(vB[1], vB[0]), 50)
arc_radius = 2.0
ax3.plot(arc_radius * np.cos(theta), arc_radius * np.sin(theta), color="darkorange", linestyle="-", linewidth=2.5)
ax3.text(1.5, 2.5, "Cosine Distance (θ)\n(Measures Strategic Angle Only)", color="darkorange", fontsize=9, fontweight="semibold", bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.7))

# Layout configurations
ax3.set_xlim(-1, 8)
ax3.set_ylim(-1, 14)
ax3.axhline(0, color="black", linewidth=0.8, alpha=0.5)
ax3.axvline(0, color="black", linewidth=0.8, alpha=0.5)
ax3.set_xlabel("Situational Rate X")
ax3.set_ylabel("Situational Rate Y")
ax3.set_title("Distance Metrics: Euclidean Magnitude Space vs. Cosine Angular Space", fontsize=12, fontweight="bold", pad=12)
ax3.grid(True, linestyle="--", alpha=0.3)
ax3.legend(loc="upper right", frameon=True)
plt.tight_layout()
plt.savefig("placeholder_cluster_theory2.png", dpi=150, bbox_inches="tight")
plt.show()